# 17 Differential Abundance and Age

We use ANCOM-BC2 to test whether fungal genus abundance is associated with age in years (`age_years_sample`). We use the unrarefied genus count table from notebook 14 and keep the same samples for the unadjusted and adjusted models. The adjusted model accounts for sex and settlement type, as in notebook 10.

For each taxon, the log fold change describes the estimated difference in log abundance between participants who are one year apart in age. A positive value means higher estimated abundance in the older participant. A negative value means lower estimated abundance. 

## 1. Setup

### 1.1 Imports

In [1]:
import pandas as pd
import qiime2 as q2
from qiime2 import Artifact, Metadata

In [3]:
metadata = "../data/raw/Final_metadata_with_GHSL_exclude_other_sex.tsv"
in_dir = "../data/processed_ignored/14_differential_abundance_setup"
out_dir = "../data/processed_ignored/17_differential_abundance_age"
vis_dir = "../data/processed/17_differential_abundance_age"

unadjusted_dir = f"{vis_dir}/unadjusted_ancombc2" 
adjusted_dir = f"{vis_dir}/adjusted_ancombc2"

genus_table = f"{in_dir}/table_fungi_l6.qza"
age_table = f"{out_dir}/table_fungi_l6_age.qza"

age_unadjusted_qza = f"{out_dir}/ancombc2_age_unadjusted.qza"
age_adjusted_qza = f"{out_dir}/ancombc2_age_adjusted.qza"

age_unadjusted_tabulate = f"{unadjusted_dir}/tabulate_unadjusted_age.qzv"
age_adjusted_tabulate = f"{adjusted_dir}/tabulate_adjusted_age.qzv"

age_unadjusted_barplot = f"{unadjusted_dir}/barplot_unadjusted_age.qzv"
age_adjusted_barplot = f"{adjusted_dir}/barplot_adjusted_age.qzv"

!mkdir -p "$out_dir" "$vis_dir"
!mkdir -p "$unadjusted_dir" "$adjusted_dir"

### 1.3 Load inputs

We load the genus count table from notebook 14 and the cleaned sample metadata used in notebook 09.

In [4]:
genus_artifact = Artifact.load(genus_table)
sample_metadata = Metadata.load(metadata)

print("Table type:", genus_artifact.type)
print("Samples in metadata:", sample_metadata.id_count)

Table type: FeatureTable[Frequency]
Samples in metadata: 149


## 2. Select samples

### 2.1 Keep complete cases

We keep samples with settlement type, age, and diet information so both models use the same samples.

In [6]:
!qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "[Settlement_Type] IS NOT NULL AND [sex_sample] IS NOT NULL AND [age_years_sample] IS NOT NULL" \
    --o-filtered-table "$age_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/17_differential_abundance_age/table_fungi_l6_age.qza


### 2.2 Summarize selected samples

We summarize the numeric age values after filtering for complete settlement, sex, and age information.

In [8]:
age_counts = Artifact.load(age_table).view(pd.DataFrame)
selected_metadata = sample_metadata.to_dataframe().loc[age_counts.index]

print("Selected samples:", len(selected_metadata))
display(selected_metadata["age_years_sample"].describe())

Selected samples: 141


count    141.000000
mean      48.645390
std       18.723223
min        2.000000
25%       36.000000
50%       53.000000
75%       63.000000
max       82.000000
Name: age_years_sample, dtype: float64

## 3. Run ANCOM-BC2

We use ANCOM-BC2 because some fungal genera were detected in only a few samples. When a genus has many zero counts, a result can depend on the small value used in place of zero. ANCOM-BC2 checks this by trying several small values and seeing whether the result stays significant. 

We add `--p-diff-robust` to both models to see which results stay significant when ANCOM-BC2 tries different small values for the zero counts.

### 3.1 Unadjusted ANCOM-BC2

We test whether fungal genus abundance correlates with age. The log fold change tells us how much the predicted log abundance differs per year of age.

In [11]:
print(Metadata.load(metadata).get_column("age_years_sample").type)

numeric


In [9]:
!qiime composition ancombc2 \
    --i-table "$age_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "age_years_sample" \
    --p-diff-robust \
    --o-ancombc2-output "$age_unadjusted_qza"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/17_differential_abundance_age/ancombc2_age_unadjusted.qza


In [12]:
!qiime composition tabulate \
    --i-data "$age_unadjusted_qza" \
    --o-visualization "$age_unadjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$age_unadjusted_qza" \
    --o-visualization "$age_unadjusted_barplot"


Saved Visualization to: ../data/processed/17_differential_abundance_age/unadjusted_ancombc2/tabulate_unadjusted_age.qzv
Saved Visualization to: ../data/processed/17_differential_abundance_age/unadjusted_ancombc2/barplot_unadjusted_age.qzv


In [13]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(age_unadjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

age = "age_years_sample"

results = pd.DataFrame({
    "log_fold_change_per_year": tables["lfc"][age],
    "q_value": tables["q"][age],
    "diff_robust": tables["diff_robust"][age],
})

significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")

print(f"Unadjusted age model — {len(significant)} taxa with q < 0.05")
display(significant)

Unadjusted age model — 6 taxa with q < 0.05


,log_fold_change_per_year,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,0.071533,0.000007,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Metschnikowiaceae;g__Clavispora,-0.073712,0.000010,False
k__Fungi;p__Basidiomycota;c__Microbotryomycetes;o__Sporidiobolales;f__Sporidiobolaceae;g__Rhodotorula,-0.056255,0.001305,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Kluyveromyces,0.049114,0.003895,False
k__Fungi;p__Ascomycota;c__Dothideomycetes;o__Cladosporiales;f__Cladosporiaceae;g__Cladosporium,-0.045661,0.010796,False
k__Fungi;p__Mucoromycota;c__Mucoromycetes;o__Mucorales;f__Mucoraceae;g__Mucor,-0.042767,0.014819,False


#### Unadjusted results

The analysis included 141 participants aged 2–82 years (median 53 years). Six taxa had q < 0.05 for their association with age.

| Taxon | Log fold change per year | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae* | +0.0715 | 0.000007 | False |
| *Clavispora* | −0.0737 | 0.000010 | False |
| *Rhodotorula* | −0.0563 | 0.001305 | False |
| *Kluyveromyces* | +0.0491 | 0.003895 | False |
| *Cladosporium* | −0.0457 | 0.010796 | False |
| *Mucor* | −0.0428 | 0.014819 | False |

*genus unidentified

Older participants tended to have higher estimated abundance of the unidentified Dipodascaceae taxon and *Kluyveromyces*, and lower estimated abundance of *Clavispora*, *Rhodotorula*, *Cladosporium*, and *Mucor*. None of the six age associations passed the sensitivity check for zero counts, so we interpret these patterns cautiously.

### 3.2 Adjusted ANCOM-BC2

We observe age-related shifts in mycobial abundance after accounting for sex and settlement. We use the same 141 samples as in the unadjusted model.

In [14]:
!qiime composition ancombc2 \
    --i-table "$age_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "Settlement_Type + sex_sample + age_years_sample" \
    --p-diff-robust \
    --o-ancombc2-output "$age_adjusted_qza"

!qiime composition tabulate \
    --i-data "$age_adjusted_qza" \
    --o-visualization "$age_adjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$age_adjusted_qza" \
    --o-visualization "$age_adjusted_barplot"


Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/17_differential_abundance_age/ancombc2_age_adjusted.qza
Saved Visualization to: ../data/processed/17_differential_abundance_age/adjusted_ancombc2/tabulate_adjusted_age.qzv
Saved Visualization to: ../data/processed/17_differential_abundance_age/adjusted_ancombc2/barplot_adjusted_age.qzv


In [16]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(age_adjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

age = "age_years_sample"

results = pd.DataFrame({
    "log_fold_change_per_year": tables["lfc"][age],
    "q_value": tables["q"][age],
    "diff_robust": tables["diff_robust"][age],
})

significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")

print(f"Adjusted age model — {len(significant)} taxa with q < 0.05")
display(significant)

Adjusted age model — 4 taxa with q < 0.05


,log_fold_change_per_year,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Dipodascaceae;__,0.089995,8.500000e-09,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Kluyveromyces,0.086211,2.137860e-05,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Metschnikowiaceae;g__Clavispora,-0.056682,1.340229e-04,False
k__Fungi;p__Mucoromycota;c__Mucoromycetes;o__Mucorales;f__Mucoraceae;g__Mucor,-0.038252,1.303804e-02,False


#### Adjusted results

| Taxon | Log fold change per year | q-value | diff_robust |
|---|---:|---:|:---:|
| Dipodascaceae* | +0.0900 | 8.50 × 10⁻⁹ | False |
| *Kluyveromyces* | +0.0862 | 0.0000214 | False |
| *Clavispora* | −0.0567 | 0.000134 | False |
| *Mucor* | −0.0383 | 0.013038 | False |

*genus unidentified

Four of the six taxa from the unadjusted model still had q < 0.05 after adjustment, all in the same direction. Older participants tended to have higher estimated abundance of the unidentified Dipodascaceae taxon and *Kluyveromyces*, and lower estimated abundance of *Clavispora* and *Mucor*. None of the four associations passed the sensitivity check for zero counts, so we interpret these age trends cautiously.

## 4. Discussion

Four taxa showed the same age trend before and after adjustment. *Kluyveromyces* and the unidentified Dipodascaceae taxon tended to be more abundant in older participants, while *Clavispora* and *Mucor* tended to be less abundant. None of these trends passed the sensitivity check for zero counts, so we interpret them cautiously.